# Informed search & A* — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
DIRS=[(1,0),(-1,0),(0,1),(0,-1)]
def neighbors(p, shape=(5,5), walls=set()):
    for dr,dc in DIRS:
        q=(p[0]+dr,p[1]+dc)
        if 0<=q[0]<shape[0] and 0<=q[1]<shape[1] and q not in walls: yield q
def reconstruct(parent, goal):
    path=[]; x=goal
    while x is not None: path.append(x); x=parent.get(x)
    return path[::-1]
def draw(vals, title):
    plt.figure(figsize=(5,3)); plt.plot(vals,'-o'); plt.title(title)
def draw_grid(path=(), explored=(), frontier=(), walls=set(), start=(0,0), goal=(4,4), shape=(5,5), title='grid'):
    img=np.zeros(shape)
    for r,c in walls: img[r,c]=-1
    for r,c in explored: img[r,c]=.35
    for r,c in frontier: img[r,c]=.65
    for r,c in path: img[r,c]=1
    plt.figure(figsize=(4,4)); plt.imshow(img,cmap='viridis',vmin=-1,vmax=1); plt.scatter([start[1]],[start[0]],c='w',edgecolor='k'); plt.scatter([goal[1]],[goal[0]],c='r',edgecolor='k'); plt.title(title); plt.xticks(range(shape[1])); plt.yticks(range(shape[0])); plt.grid(color='w',alpha=.3)
print('setup ok')

## Cost so far plus estimated cost to go
A* is uniform-cost search with a lower-bound heuristic added to the priority.

In [ ]:
goal=(4,4); vals=np.array([[abs(r-goal[0])+abs(c-goal[1]) for c in range(5)] for r in range(5)])
plt.figure(figsize=(4,4)); plt.imshow(vals,cmap='magma'); plt.colorbar(); plt.title('Manhattan heuristic')
assert vals[0,0]==8 and vals[4,4]==0

In [ ]:
import heapq
walls={(1,1),(1,2),(2,1),(3,3)}; s=(0,0); g=(4,4); H=lambda p: abs(p[0]-g[0])+abs(p[1]-g[1]); pq=[(H(s),0,s)]; dist={s:0}; par={s:None}; ex=[]
while pq:
 f,d,u=heapq.heappop(pq)
 if d!=dist[u]: continue
 ex.append(u)
 if u==g: break
 for v in neighbors(u,walls=walls):
  nd=d+1
  if nd<dist.get(v,999): dist[v]=nd; par[v]=u; heapq.heappush(pq,(nd+H(v),nd,v))
p=reconstruct(par,g); draw_grid(p,ex,[x for _,_,x in pq],walls=walls,title=f'A* length {len(p)-1}')
assert dist[g]==8 and len(p)-1==8

In [ ]:
def astar(scale):
 import heapq
 h=lambda p: scale*(abs(p[0]-g[0])+abs(p[1]-g[1])); pq=[(h(s),0,s)]; d={s:0}; par={s:None}; ex=[]
 while pq:
  f,du,u=heapq.heappop(pq)
  if du!=d[u]: continue
  ex.append(u)
  if u==g: break
  for v in neighbors(u,walls=walls):
   nd=du+1
   if nd<d.get(v,999): d[v]=nd; par[v]=u; heapq.heappush(pq,(nd+h(v),nd,v))
 return ex,reconstruct(par,g)
e0,p0=astar(0); e1,p1=astar(1); plt.figure(figsize=(5,3)); plt.bar(['h=0','Manhattan'],[len(e0),len(e1)]); plt.title('heuristic shrinks search')
assert len(p0)==len(p1) and len(e1)<=len(e0)

In [ ]:
m=[]
for r in range(5):
 for c in range(5):
  for v in neighbors((r,c),walls=walls): m.append(1+H(v)-H((r,c)))
plt.figure(figsize=(5,3)); plt.hist(m,bins=[0,1,2,3]); plt.title('consistency margins')
assert min(m)==0 and all(x>=0 for x in m)

In [ ]:
e2,p2=astar(2); draw_grid(p2,e2,walls=walls,title=f'weighted A* expansions {len(e2)}')
assert len(p2)-1==8 and len(e2)<=len(e1)